#### implement Yolov9 mit on colab

##### we changed to function like calling, in order to run on colab smoothly

#### clone, install

In [ ]:
!git clone https://github.com/KuoYuChang/YOLO.git

Cloning into 'YOLO'...
remote: Enumerating objects: 3870, done.
remote: Counting objects: 100% (1658/1658), done.
remote: Compressing objects: 100% (327/327), done.
remote: Total 3870 (delta 1505), reused 1360 (delta 1331), pack-reused 2212 (from 1)
Receiving objects: 100% (3870/3870), 1.61 MiB | 13.37 MiB/s, done.
Resolving deltas: 100% (2544/2544), done.


In [ ]:
%cd YOLO

/content/YOLO


In [ ]:
%ls

demo/    examples/       README.md             tests/
docker/  LICENSE         requirements-dev.txt  yolo/
docs/    pyproject.toml  requirements.txt


In [ ]:
!pip install -r requirements.txt

#### checkout to colab friendly branch

In [ ]:
!git checkout CloudNB

Branch 'CloudNB' set up to track remote branch 'CloudNB' from 'origin'.
Switched to a new branch 'CloudNB'


#### import libraries

In [ ]:
### try function call
from hydra import compose, initialize
from omegaconf import OmegaConf

from yolo.lazy_pure import run_solver

#### Instead of calling py file
#### Here we changed to call functions

#### inference
#### input images to model, output result images

#### Note that config_fd_path may be different, due to colab status
#### seems that default location be in /tmp
#### but our file usually in content/YOLO
#### trail and error

In [ ]:

config_fd_path = "../../content/YOLO/yolo/config"

with initialize(config_path=config_fd_path, version_base=None):
    cfg = compose(config_name="config_pure.yaml", overrides=["task=inference",
                                                            "name=DemoResult",
                                                            "device=cuda", # cpu or cuda
                                                            "model=v9-s",
                                                            "task.nms.min_confidence=0.5",
                                                            "task.data.source=demo/images/inference",
                                                            "+quite=False",
                                                            #"weight=weights/yolov9s_test.pt"
                                                            ])
    solver = run_solver(cfg, print_frac=0.05)

#### validation

In [ ]:
config_fd_path = "../../content/YOLO/yolo/config"

with initialize(config_path=config_fd_path, version_base=None):
    cfg = compose(config_name="config_pure.yaml", overrides=["task=validation",
                                                            "task.data.batch_size=1",
                                                            "dataset=coco",
                                                            #"dataset=football",
                                                            "name=ValResult",
                                                            "device=cuda", # cpu or cuda
                                                            "model=v9-s",
                                                            "use_wandb=False",
                                                            "cpu_num=1",

                                                            #"task.nms.min_confidence=0.001",

                                                            #"+limit_val_batches=4"
                                                          ])

    solver = run_solver(cfg, print_frac=0.05)

##### COCO validation set result: PyCOCO/AP @ .5:.95: 0.4611353576183319, PyCOCO/AP @ .5: 0.6224560141563416
##### close to original work evaluations
##### ------------------------------------------------------------------------------------

#### Training

#### here we do transfer learning with pretrained weight
#### * prepare dataset
#### * arrange folders as coco structure
#### * prepare dataset config

In [ ]:
%mkdir data
%cd data

mkdir: cannot create directory ‘data’: File exists
/content/YOLO/data


#### Here we download football dataset from roboflow

* visit roboflow [football dataset](https://universe.roboflow.com/roboflow-jvuqo/football-players-detection-3zvbc)
* click Dataset, choose v8 2024-03-01
* Choose COCO JSON format
* Download, get download snippet code

In [ ]:
!pip install -q roboflow

In [ ]:
#!pip install roboflow

from roboflow import Roboflow

# hide this line
rf = Roboflow(api_key="Your Api Key")

project = rf.workspace("roboflow-jvuqo").project("football-players-detection-3zvbc")
version = project.version(8)
dataset = version.download("coco")

#### data folder architecture

#### coco format detection dataset
#### note that name of annotation json files need fix as following
```
football
|
├── annotations
|   ├── instances_test.json
|   ├── instances_test.json
|   └── instances_test.json
└── images
    ├── test
    │   ├── 095.png
    ├── train
    │   ├── 000.png
    │   ├── 001.png
    │   └── 002.png
    └── valid
        ├── 080.png
        └── 081.png
```

#### We're moving files to match architecture

In [ ]:
%cd football-players-detection-8

/content/YOLO/data/football-players-detection-8


In [ ]:
%ls

README.dataset.txt  README.roboflow.txt  test/  train/  valid/


In [ ]:
%mkdir annotations
%mkdir images

In [ ]:
%mv train/_annotations.coco.json annotations/instances_train.json
%mv valid/_annotations.coco.json annotations/instances_valid.json
%mv test/_annotations.coco.json annotations/instances_test.json

In [ ]:
%mv train/ images/
%mv valid/ images/
%mv test/ images/

In [ ]:
%ls

annotations/  images/  README.dataset.txt  README.roboflow.txt


In [ ]:
%ls annotations

instances_test.json  instances_train.json  instances_valid.json


In [ ]:
%ls images

test/  train/  valid/


In [ ]:
# done, back to YOLO/
%cd ../..
%ls

/content/YOLO
data/    docs/      pyproject.toml        requirements.txt  weights/
demo/    examples/  README.md             runs/             yolo/
docker/  LICENSE    requirements-dev.txt  tests/


#### Prepare football data config

#### Here we recommend adding class 0th as dummy
#### so became n+1 class: 0 to n
#### prevent if not providing categories mapping in annotation jsons

In [ ]:
%%writefile yolo/config/dataset/football.yaml
path: data/football-players-detection-8
train: train
validation: valid

class_num: 5
class_list: ['dummy', 'ball', 'goalkeeper', 'player', 'referee']

auto_download:

Writing yolo/config/dataset/football.yaml


#### train
#### fine-tune/ transfer learning on football dataset

In [ ]:
config_fd_path = "../../content/YOLO/yolo/config"

with initialize(config_path=config_fd_path, version_base=None):
    cfg = compose(config_name="config_pure.yaml", overrides=["task=train_pure",

                                                            "task.epoch=10",
                                                            "task.data.batch_size=2",
                                                            "dataset=football",
                                                            #"dataset=coco",
                                                            "name=TrainResult",
                                                            "device=cuda", # cpu or cuda
                                                            "model=v9-s",
                                                            "use_wandb=False",
                                                            "cpu_num=1",

                                                            #"task.optimizer.args.lr=0.05",

                                                            "task.data.data_augment.Mosaic=0.25", #experimental
                                                            "task.data.data_augment.RandomCrop=0.0", # too large area cropped, turn off

                                                            #"task.data.data_augment.MixUp=0.15",
                                                            "task.data.data_augment.HorizontalFlip=0.5",

                                                            #"+limit_val_batches=4"
                                                            "precision16=False",
                                                            "task.optimizer.clip_norm=2",

                                                            "task.fine_tune.lists=[]",
                                                            #"task.fine_tune.lists=[0, 1, 2, 3, 4, 5, 6, 7, 8]",
                                                            #"task.fine_tune.lists=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]",
                                                          ])

    #print(OmegaConf.to_yaml(cfg))
    solver = run_solver(cfg, print_frac=0.05)

In [ ]:
#### rename trained model
%cp runs/train/TrainResult/weights/v9-s9.pt weights/yolov9s_football.pt

#### validate on test folder
#### remember change dataset val parameter to test folder

In [ ]:
config_fd_path = "../../content/YOLO/yolo/config"

with initialize(config_path=config_fd_path, version_base=None):
    cfg = compose(config_name="config_pure.yaml", overrides=["task=validation",
                                                            "task.data.batch_size=1",
                                                            #"dataset=coco",
                                                            "dataset=football",

                                                            ## important, change val image folder
                                                            "dataset.validation=test",

                                                            "name=ValResult",
                                                            "device=cuda", # cpu or cuda
                                                            "model=v9-s",
                                                            "use_wandb=False",
                                                            "cpu_num=1",

                                                            #"task.nms.min_confidence=0.001",

                                                            #"+limit_val_batches=4",
                                                            "weight=weights/yolov9s_football.pt",
                                                          ])

    solver = run_solver(cfg, print_frac=0.05)

##### football testset result: PyCOCO/AP @ .5:.95: 0.3963283598423004, PyCOCO/AP @ .5: 0.6200857758522034

#### inference on test folder

In [ ]:
config_fd_path = "../../content/YOLO/yolo/config"

with initialize(config_path=config_fd_path, version_base=None):
    cfg = compose(config_name="config.yaml", overrides=["task=inference",
                                                        "name=FootballResult",
                                                        "dataset=football",

                                                        "device=cuda", # cpu or cuda
                                                        "model=v9-s",
                                                        "task.nms.min_confidence=0.5",
                                                        "task.data.source=data/football-players-detection-8/images/test",
                                                        "+quite=False",
                                                        "weight=weights/yolov9s_football.pt",
                                                        ])
    solver = run_solver(cfg, print_frac=0.05)

In [ ]:
%cd runs/inference

/content/YOLO/runs/inference


In [ ]:
!zip -r FootballResult.zip FootballResult

  adding: FootballResult/ (stored 0%)
  adding: FootballResult/frame010.png (deflated 0%)
  adding: FootballResult/frame000.png (deflated 0%)
  adding: FootballResult/frame004.png (deflated 0%)
  adding: FootballResult/frame009.png (deflated 0%)
  adding: FootballResult/frame002.png (deflated 0%)
  adding: FootballResult/frame001.png (deflated 0%)
  adding: FootballResult/frame007.png (deflated 0%)
  adding: FootballResult/frame008.png (deflated 0%)
  adding: FootballResult/frame005.png (deflated 0%)
  adding: FootballResult/output.log (deflated 77%)
  adding: FootballResult/frame003.png (deflated 0%)
  adding: FootballResult/frame011.png (deflated 0%)
  adding: FootballResult/frame012.png (deflated 0%)
  adding: FootballResult/frame006.png (deflated 0%)


In [ ]:
%cd ../..

/content/YOLO


In [ ]:
%ls

data/    docs/      pyproject.toml        requirements.txt  weights/
demo/    examples/  README.md             runs/             yolo/
docker/  LICENSE    requirements-dev.txt  tests/
